# Verification workflow demo for v0.3.3-alpha

This notebook executes deterministic format, checksum and policy checks on synthetic contribution events using the same validation vocabulary as the manuscript, SI, JSON schemas and output tables. It does not query the live ORCID or DOI registries and does not authenticate a scientific claim.


In [ ]:
from pathlib import Path
import csv, json, re

events = json.loads(Path('data/example_contributions.json').read_text(encoding='utf-8'))
len(events)


## Deterministic structural and boundary checks

The notebook records metadata, evidence-file, file-integrity, source-link and scientific-assessment states separately. It never maps file integrity to scientific review.

In [ ]:
ORCID_RE = re.compile(r'^\d{4}-\d{4}-\d{4}-\d{3}[0-9X]$')
DOI_RE = re.compile(r'^10\.[^\s/]+/.+')

def orcid_checksum_valid(value):
    compact = value.replace('-', '')
    if not ORCID_RE.fullmatch(value):
        return False
    total = 0
    for char in compact[:15]:
        total = (total + int(char)) * 2
    result = (12 - (total % 11)) % 11
    expected = 'X' if result == 10 else str(result)
    return compact[-1] == expected

def inspect_event(event):
    v = event['validation']
    verifier = v['verifier']
    contributor_orcid = event['contributor']['orcid']
    checks = {
        'orcid_format_matches': bool(ORCID_RE.fullmatch(contributor_orcid)),
        'orcid_checksum_valid': orcid_checksum_valid(contributor_orcid),
        'doi_format_matches': bool(DOI_RE.fullmatch(event['research_object'].get('doi', ''))),
        'repository_url_format_matches': event['research_object'].get('repository_url', '').startswith('https://'),
        'evidence_link_recorded': len(event.get('evidence', {}).get('links', [])) > 0,
        'self_verification_rejected': not (verifier.get('type') == 'person' and verifier.get('identifier') == contributor_orcid),
        'is_non_transferable': event['issued_credential'].get('non_transferable') is True,
        'is_locked': event['issued_credential'].get('locked') is True,
    }
    return {
        'event_id': event['event_id'],
        'contribution_type': event['contribution_type'],
        'metadata_status': v['metadata_status'],
        'evidence_file_status': v['evidence_file_status'],
        'file_integrity_status': v['file_integrity_status'],
        'source_link_status': v['source_link_status'],
        'scientific_assessment_status': v['scientific_assessment']['status'],
        **checks,
        'checks_passed': sum(checks.values()),
        'checks_total': len(checks),
    }

verification_rows = [inspect_event(e) for e in events]
verification_rows[:2]


## Export reviewer-inspectable results

In [ ]:
out = Path('outputs/verification_results.csv')
out.parent.mkdir(exist_ok=True)
with out.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(verification_rows[0].keys()))
    writer.writeheader()
    writer.writerows(verification_rows)
assert all(row['scientific_assessment_status'] == 'not_reviewed' for row in verification_rows)
assert all(row['self_verification_rejected'] for row in verification_rows)
print(f'Wrote {out} with {len(verification_rows)} synthetic rows')

## Interpretation and limitations

The output records format and checksum checks only. `orcid_checksum_valid` does not mean that an ORCID Registry record exists, and `doi_format_matches` does not mean that a DOI resolves. `source_link_recorded` does not mean that a remote source was resolved, while `file_integrity_confirmed` does not mean that a scientific claim was reviewed or found true.
